# Homework 4

This Notebook builds on the DCOPF model introduced in [Notebook 6](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks) and incorporates some elements of Economic Dispatch introduced in [Notebook 4](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks).

First, load (or install if necessary) a set of packages you'll need for this assignment...

In [1]:
using JuMP
using HiGHS
using DataFrames
using CSV
using Plots; plotly();

## Question 1: Modifying IEEE-14

**A. Increased generation costs**

Copy the IEEE 14 bus system and DCOPF solver function from Notebook 6. Since we neglect the resistance for the purpose of solving the DC-OPF, approximate the susceptance as:

$$
B = \frac{1}{X}
$$


In addition, add the following line to the return call of the function:
```julia
status = termination_status(DCOPF)
```
This tells you the solver termination status for the problem: e.g. was an optimal solution found, was the solution infeasible, was it unbounded, etc.

Make the following change to the system:

- Increase the variable cost of Generator 1 to \$30 / MWh

Run the DCOPF and output generation, flows, and prices.

In [2]:
datadir = joinpath("..","Notebooks","ieee_test_cases") 
gens = CSV.read(joinpath(datadir,"Gen14.csv"), DataFrame);
lines = CSV.read(joinpath(datadir,"Tran14.csv"), DataFrame);
loads = CSV.read(joinpath(datadir,"Load14.csv"), DataFrame);

# Rename all columns to lowercase (by convention)
for f in [gens, lines, loads]
    rename!(f,lowercase.(names(f)))
end

# create generator ids 
gens.id = 1:nrow(gens);

# create line ids 
lines.id = 1:nrow(lines);
# add set of rows for reverse direction with same parameters
lines2 = copy(lines)
lines2.f = lines2.fromnode
lines2.fromnode = lines.tonode
lines2.tonode = lines2.f
lines2 = lines2[:,names(lines)]
append!(lines,lines2)

# calculate simple susceptance, ignoring resistance as earlier 

lines.b = 1 ./ lines.reactance

# keep only a single time period
loads = loads[:,["connnode","interval-1_load"]]
rename!(loads,"interval-1_load" => "demand");

In [3]:
#=
Function to solve DC OPF problem using IEEE test cases
Inputs:
    gen_info -- dataframe with generator info
    line_info -- dataframe with transmission lines info
    loads  -- dataframe with load info
=#
function dcopf_ieee(gens, lines, loads)
    DCOPF = Model(HiGHS.Optimizer) # You could use Clp as well, with Clp.Optimizer
    
    # Define sets based on data
      # Set of generator buses
    G = gens.connnode
    
      # Set of all nodes
    N = sort(union(unique(lines.fromnode), 
            unique(lines.tonode)))
    
      # sets J_i and G_i will be described using dataframe indexing below

    # Define per unit base units for the system 
    # used to convert from per unit values to standard unit
    # values (e.g. p.u. power flows to MW/MVA)
    baseMVA = 100 # base MVA is 100 MVA for this system
    
    # Decision variables   
    @variables(DCOPF, begin
        GEN[N]  >= 0     # generation        
        # Note: we assume Pmin = 0 for all resources for simplicty here
        THETA[N]         # voltage phase angle of bus
        FLOW[N,N]        # flows between all pairs of nodes
    end)
    
    # Create slack bus with reference angle = 0; use bus 1 with generator
    fix(THETA[1],0)
                
    # Objective function
    @objective(DCOPF, Min, 
        sum( gens[g,:c1] * GEN[g] for g in G)
    )
    
    # Supply demand balances
    @constraint(DCOPF, cBalance[i in N], 
        sum(GEN[g] for g in gens[gens.connnode .== i,:connnode]) 
            + sum(load for load in loads[loads.connnode .== i,:demand]) 
        == sum(FLOW[i,j] for j in lines[lines.fromnode .== i,:tonode])
    )

    # Max generation constraint
    @constraint(DCOPF, cMaxGen[g in G],
                    GEN[g] <= gens[g,:pgmax])
    
    # Flow constraints on each branch 
    @constraint(DCOPF, cLineFlows[l in 1:nrow(lines)],
            FLOW[lines[l,:fromnode],lines[l,:tonode]] == 
            baseMVA * lines[l,:b] * 
            (THETA[lines[l,:fromnode]] - THETA[lines[l,:tonode]])
    )
    
    # Max line flow constraints
    @constraint(DCOPF, cLineLimits[l in 1:nrow(lines)], 
            FLOW[lines[l,:fromnode],lines[l,:tonode]] <=
            lines[l,:capacity]
    ) 


    # Solve statement (! indicates runs in place)
    optimize!(DCOPF)

    # Output variables
    generation = DataFrame(
        node = gens.connnode,
        gen = value.(GEN).data[gens.connnode]
        )
    
    angles = value.(THETA).data
    
    flows = DataFrame(
        fbus = lines.fromnode,
        tbus = lines.tonode,
        flow = baseMVA * lines.b .* (angles[lines.fromnode] .- 
                        angles[lines.tonode]))
    
    # We output the marginal values of the demand constraints, 
    # which will in fact be the prices to deliver power at a given bus.
    prices = DataFrame(
        node = N,
        value = dual.(cBalance).data)
    
    # Return the solution and objective as named tuple
    return (
        generation = generation, 
        angles,
        flows,
        prices,
        cost = objective_value(DCOPF),
        status = termination_status(DCOPF)
    )
end

dcopf_ieee (generic function with 1 method)

In [4]:
# increase the variable cost of Generator 1 to $30/MWh
gens.c1[1] = 30

30

In [5]:
solution = dcopf_ieee(gens, lines, loads);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e+00, 2e+03]
  Cost   [2e+01, 3e+01]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 1e+04]
Presolving model
46 rows, 47 cols, 140 nonzeros  0s
31 rows, 32 cols, 109 nonzeros  0s
13 rows, 14 cols, 51 nonzeros  0s
Dependent equations search running on 8 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
8 rows, 9 cols, 30 nonzeros  0s
Presolve : Reductions: rows 8(-88); columns 9(-215); elements 30(-174)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.5694757857e+00 Pr: 8(86042.1) 0s
          8     7.0700000000e+03 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model status        : Optimal
Simplex   iterations: 8
Objective value     :  7.0700000

In [6]:
solution.generation

Row,node,gen
,Int64,Float64
1,1,119.0
2,2,140.0


In [7]:
solution.flows

Row,fbus,tbus,flow
,Int64,Int64,Float64
1,1,2,64.0779
2,1,5,54.9221
3,2,3,72.7846
4,2,4,60.9488
5,2,5,48.6446
6,3,4,-21.4154
7,4,5,-54.3377
8,4,7,29.274
9,4,9,16.7971


In [8]:
solution.prices

Row,node,value
,Int64,Float64
1,1,30.0
2,2,30.0
3,3,30.0
4,4,30.0
5,5,30.0
6,6,30.0
7,7,30.0
8,8,30.0
9,9,30.0


Regarding the above results, answer the following:

- How has generation changed compared to the default system?
- What explains the new prices?


Instead of all generation (259 MW) produced by generator 1, as in the default system, the majority of generation (140 MW) is now produced by generator 2 (up to it's maximum capacity) due to the increased variable cost of generator 1. The prices are still set by generator 1 at $30/MWh, because prices are set by the marginal generator in an essentially copper-plate system (i.e. no congestion/transmission constraints and therefore no price discrepancy across nodes).

**B. Constrained line**

Make the following changes to the system:

- Increase the variable cost of Generator 1 to \$30 / MWh
- Reduce flow limit on the line connecting 2 and 3 ($l_{23}$) to 70 MW

Run the DCOPF and output generation, flows, and prices.

In [9]:
# Reduce flow limit on line connecting 2 and 3 (l23) to 70 MW
lines[(lines.fromnode .== 2) .& (lines.tonode .== 3), :capacity] .= 70
lines[(lines.fromnode .== 3) .& (lines.tonode .== 2), :capacity] .= 70

1-element view(::Vector{Int64}, [23]) with eltype Int64:
 70

In [10]:
solution = dcopf_ieee(gens, lines, loads);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e+00, 2e+03]
  Cost   [2e+01, 3e+01]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 1e+04]
Presolving model
46 rows, 47 cols, 140 nonzeros  0s
31 rows, 32 cols, 109 nonzeros  0s
13 rows, 14 cols, 48 nonzeros  0s
9 rows, 10 cols, 30 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve : Reductions: rows 0(-96); columns 0(-224); elements 0(-204) - Reduced to empty
Solving the original LP from the solution after postsolve
Model status        : Optimal
Objective value     :  7.5791866158e+03
P-D objective error :  5.9995534636e-17
HiGHS run time      :          0.00


In [11]:
solution.generation

Row,node,gen
,Int64,Float64
1,1,220.837
2,2,38.1627


In [12]:
solution.flows

Row,fbus,tbus,flow
,Int64,Int64,Float64
1,1,2,149.42
2,1,5,71.417
3,2,3,70.0
4,2,4,55.1212
5,2,5,40.7618
6,3,4,-24.2
7,4,5,-62.4868
8,4,7,28.9798
9,4,9,16.6283


In [13]:
solution.prices

Row,node,value
,Int64,Float64
1,1,30.0
2,2,25.0
3,3,127.287
4,4,57.6793
5,5,48.8474
6,6,51.8507
7,7,56.0958
8,8,56.0958
9,9,55.2628


Regarding the above results, answer the following:

- Which node has the highest price and why?
- What is the difference in prices across $l_{23}$, also known as the congestion rent? How do you interpret this value (what is it's practical meaning?)

Node 3 has the highest price because it experiences the most congestion since l23 is capacity constrained, creating a transmission bottleneck that restricts cheaper generation from reaching node 3. The congestion rent across l23 is $102.287/MWh (127.287 - 25), representing the shadow price/marginal value of adding more capacity on the line i.e. total system costs would decrease by $102.287 if 1 MW capacity is added to the line. Looking at congestion rents tells us where additional transmission capacity would be the most financially beneficial to invest in expanding.

**C. Demand increase**

Make the following changes to the system:

- Increase the variable cost of Generator 1 to \$30 / MWh
- Reduce flow limit on the line connecting 2 and 3 ($l_{23}$) to 70 MW
- Increase demands everywhere by 5\%.

In [14]:
# Increase all demand by 5%
loads.demand = loads.demand .* 1.05;

Calculate the total available generating capacity:

In [15]:
sum(gens.pgmax)

440

Calculate the new total demand:

In [16]:
sum(loads.demand)

-271.94999999999993

Run the DCOPF and show prices.

In [17]:
solution = dcopf_ieee(gens, lines, loads)
solution.prices

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e+00, 2e+03]
  Cost   [2e+01, 3e+01]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 1e+04]
Presolving model
46 rows, 47 cols, 140 nonzeros  0s
31 rows, 32 cols, 109 nonzeros  0s
13 rows, 14 cols, 48 nonzeros  0s
9 rows, 10 cols, 30 nonzeros  0s
Problem status detected on presolve: Infeasible
Model status        : Infeasible
Objective value     :  0.0000000000e+00
HiGHS run time      :          0.00
Solving LP to try to compute dual ray
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e+00, 2e+03]
  Cost   [0e+00, 0e+00]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 1e+04]
Solving LP without presolve, or with basis, or unconstrained
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 11(271.95); Du: 0(2.92435e-11) 0s
         54   

Row,node,value
,Int64,Float64
1,1,0.0
2,2,-0.0273434
3,3,0.532033
4,4,0.151369
5,5,0.103071
6,6,0.119495
7,7,0.14271
8,8,0.14271
9,9,0.138154


**What is happening in this system?** 

Although there is more generating capacity (440 MW) than demand (272 MW), the increase in demand compounded with constrained l23 makes the case infeasible, because the transmission network cannot meet all loads (especially at node 3).

## Question 2: Linear losses

Up until now, we have ignored transmission losses. A quadratic approximation of losses is given by:

\begin{align}
LOSS_{ij} &\approx \frac{G_{ij}}{BaseMVA} (\theta_i-\theta_j)^2 \\
 & \approx \frac{1}{BaseMVA} \frac{R_{ij}}{R_{ij}^2+X_{ij}^2}(\theta_i-\theta_j)^2
\end{align}


where $G$ is the line's conductance, $R$ is the line's resistance and $X$ is the line's reactance. See the `lines` data frame for these parameters.

For our purposes, we will approximate this quadratic via:


$$
LOSS_{ij} \geq \frac{R_{ij}}{BaseMVA} \times (MaxFlow_{ij})^2 
\left(\frac{|FLOW_{ij}|}{MaxFlow_{ij}} - 0.165 \right)
$$

where $MaxFlow_{ij}=200 MW$ in this problem. Note the greater than equal sign, as we do not want to have negative losses.

This approximation is based on Fitiwi et al. (2016), "Finding a representative network losses model for large-scale transmission expansion planning with renewable energy sources," *Energy* 101: 343-358, https://doi.org/10.1016/j.energy.2016.02.015. 

Note that this is a linear approximation of transmission losses, which are actually a quadratic function of power flows. Fitiwi et al. 2016 and other papers describe piece-wise or segment-wise linear approximations of the quadratic function which provide a tighter lower bound approximation of losses, but we'll use a single linear term for this assignment. 

See Jenkins & Sepulveda et al. 2017, "Enhanced decision support for a changing electricity landscape: the GenX configurable electricity resource capacity expansion model", MIT Energy Initiative Working Paper 2017-10 http://bit.ly/GenXModel Section 5.8, for an example of a linear segment-wise approximation of quadratic transmission losses. 


**A. Code linear losses**

Reload the original data from Notebook 6 and copy the IEEE 14 bus system and DCOPF solver function from Notebook 6 into a new function `dcopf_ieee_lossy`.

Make the following changes:
- Increase the variable cost of Generator 1 to \$30 / MWh
- Change all transmission line capacities to 200 MW

Implement losses into the supply/demand balance equations. A standard way to implement absolute values in linear programming is by introducing two non-negative auxiliary variables $x^+$, $x^-$ $\geq 0$:

$$
x = x^+ - x^-
$$

and the absolute value can be represented as:

$$
|x| = x^+ + x^-
$$

(You should satisfy yourself that this equality holds.)

It makes the formulation easier if losses are added to the supply/demand balance constraint in each node by splitting losses in half between the receiving and sending end.

Indicate which equations and variables you have added and explain your steps using inline code comments (e.g. `# Comment`).

Run the lossy DCOPF and output generation, flows, losses, and prices.

In [18]:
datadir = joinpath("..","Notebooks","ieee_test_cases") 
gens = CSV.read(joinpath(datadir,"Gen14.csv"), DataFrame);
lines = CSV.read(joinpath(datadir,"Tran14.csv"), DataFrame);
loads = CSV.read(joinpath(datadir,"Load14.csv"), DataFrame);

# Rename all columns to lowercase (by convention)
for f in [gens, lines, loads]
    rename!(f,lowercase.(names(f)))
end

# create generator ids 
gens.id = 1:nrow(gens);

# create line ids 
lines.id = 1:nrow(lines);
# add set of rows for reverse direction with same parameters
lines2 = copy(lines)
lines2.f = lines2.fromnode
lines2.fromnode = lines.tonode
lines2.tonode = lines2.f
lines2 = lines2[:,names(lines)]
append!(lines,lines2)

# calculate simple susceptance, ignoring resistance as earlier 

lines.b = 1 ./ lines.reactance

# keep only a single time period
loads = loads[:,["connnode","interval-1_load"]]
rename!(loads,"interval-1_load" => "demand");

In [19]:
gens_sens = copy(gens)
gens_sens.c1[1] = 30
lines.capacity .= 200;

In [20]:
#=
Function to solve DC OPF problem using IEEE test cases
Inputs:
    gen_info -- dataframe with generator info
    line_info -- dataframe with transmission lines info
    loads  -- dataframe with load info
=#
function dcopf_ieee_lossy(gens, lines, loads)
    DCOPF = Model(HiGHS.Optimizer) # You could use Clp as well, with Clp.Optimizer
    
    # Define sets based on data
      # Set of generator buses
    G = gens.connnode
    
      # Set of all nodes
    N = sort(union(unique(lines.fromnode), 
            unique(lines.tonode)))
    
      # Set of lines ** ADDED **
    L = 1:nrow(lines)
    
      # sets J_i and G_i will be described using dataframe indexing below

    # Define per unit base units for the system 
    # used to convert from per unit values to standard unit
    # values (e.g. p.u. power flows to MW/MVA)
    baseMVA = 100 # base MVA is 100 MVA for this system
    
    # Decision variables   
    @variables(DCOPF, begin
        GEN[N]  >= 0     # generation        
        # Note: we assume Pmin = 0 for all resources for simplicty here
        THETA[N]         # voltage phase angle of bus
        FLOW[N,N]        # flows between all pairs of nodes
        FLOW_POS[N,N] >= 0 # positive flow aux variable ** ADDED **
        FLOW_NEG[N,N] >= 0 # negative flow aux variable ** ADDED **
        LOSS[N,N] >= 0 # transmission losses in each line ** ADDED **
    end)
    
    # Create slack bus with reference angle = 0; use bus 1 with generator
    fix(THETA[1],0)
                
    # Objective function
    @objective(DCOPF, Min, 
        sum(gens[g,:c1] * GEN[g] for g in G)
    )

    # Flow constraint based on aux variables ** ADDED **
    @constraint(DCOPF, cFlowAux[l in L],
            FLOW[lines[l,:fromnode],lines[l,:tonode]] ==
            FLOW_POS[lines[l,:fromnode],lines[l,:tonode]] -
            FLOW_NEG[lines[l,:fromnode],lines[l,:tonode]] 
    )

    # Loss constraint (linear approximation) ** ADDED **
    @constraint(DCOPF, cLossLinear[l in L],
            LOSS[lines[l,:fromnode],lines[l,:tonode]] >= (lines[l,:resistance] / baseMVA) *
                        (lines[l,:capacity]^2) *
                        ((FLOW_POS[lines[l,:fromnode],lines[l,:tonode]] + FLOW_NEG[lines[l,:fromnode],lines[l,:tonode]])/lines[l,:capacity] - 0.165)
    )
    
    # Supply demand balances
    @constraint(DCOPF, cBalance[i in N], 
        sum(GEN[g] for g in gens[gens.connnode .== i,:connnode]) 
            + sum(load for load in loads[loads.connnode .== i,:demand]) 
            #- sum(0.5 * LOSS[j,i] for j in lines[lines.tonode .== i,:fromnode]) 
        == sum(FLOW[i,j] for j in lines[lines.fromnode .== i,:tonode])
            + sum(0.5 * LOSS[i,j] for j in lines[lines.fromnode .== i,:tonode]) # split losses in half between sending and receiving node
    )

    # Max generation constraint
    @constraint(DCOPF, cMaxGen[g in G],
                    GEN[g] <= gens[g,:pgmax])
    
    # Flow constraints on each branch 
    @constraint(DCOPF, cLineFlows[l in L],
            FLOW[lines[l,:fromnode],lines[l,:tonode]] == 
            baseMVA * lines[l,:b] * 
            (THETA[lines[l,:fromnode]] - THETA[lines[l,:tonode]])
    )
    
    # Max line flow constraints
    @constraint(DCOPF, cLineLimits[l in L], 
            FLOW[lines[l,:fromnode],lines[l,:tonode]] <=
            lines[l,:capacity]
    ) 

    # Solve statement (! indicates runs in place)
    optimize!(DCOPF)

    # Output variables
    generation = DataFrame(
        node = gens.connnode,
        gen = value.(GEN).data[gens.connnode]
        )
    
    angles = value.(THETA).data
    
    flows = DataFrame(
        fbus = lines.fromnode,
        tbus = lines.tonode,
        flow = baseMVA * lines.b .* (angles[lines.fromnode] .- 
                        angles[lines.tonode]))
    
    # We output the marginal values of the demand constraints, 
    # which will in fact be the prices to deliver power at a given bus.
    prices = DataFrame(
        node = N,
        value = dual.(cBalance).data)
    
    # losses output ** ADDED **
    losses = DataFrame(
        fbus = lines.fromnode,
        tbus = lines.tonode,
        loss = [value(LOSS[lines[l,:fromnode], lines[l,:tonode]]) for l in L]
    )

    # Return the solution and objective as named tuple
    return (
        generation = generation, 
        angles,
        flows,
        prices,
        losses,
        cost = objective_value(DCOPF),
        status = termination_status(DCOPF)
    )
end

dcopf_ieee_lossy (generic function with 1 method)

In [21]:
solution_lossy = dcopf_ieee_lossy(gens_sens, lines, loads);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 176 rows; 812 cols; 464 nonzeros
Coefficient ranges:
  Matrix [3e-02, 2e+03]
  Cost   [2e+01, 3e+01]
  Bound  [0e+00, 0e+00]
  RHS    [9e-01, 3e+02]
Presolving model
122 rows, 167 cols, 397 nonzeros  0s
89 rows, 131 cols, 334 nonzeros  0s
Dependent equations search running on 50 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
80 rows, 114 cols, 311 nonzeros  0s
Presolve : Reductions: rows 80(-96); columns 114(-698); elements 311(-153)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.9717651023e-02 Pr: 49(8101.97) 0s
         62     7.4896058230e+03 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model status        : Optimal
Simplex   iterations: 62
Objective value     :  7.4896058230e+03
P-D objecti

In [22]:
solution_lossy.generation

Row,node,gen
,Int64,Float64
1,1,132.987
2,2,140.0


In [23]:
solution_lossy.flows

Row,fbus,tbus,flow
,Int64,Int64,Float64
1,1,2,72.9149
2,1,5,57.9503
3,2,3,74.2381
4,2,4,62.1111
5,2,5,49.5219
6,3,4,-21.8997
7,4,5,-55.5821
8,4,7,29.2291
9,4,9,16.7713


In [24]:
solution_lossy.losses

Row,fbus,tbus,loss
,Int64,Int64,Float64
1,1,2,1.5471
2,1,5,2.69613
3,2,3,3.87556
4,2,4,3.38329
5,2,5,1.88184
6,3,4,0.0
7,4,5,0.602942
8,4,7,0.0
9,4,9,0.0


In [25]:
solution_lossy.prices

Row,node,value
,Int64,Float64
1,1,30.0
2,2,31.0351
3,3,34.4923
4,4,34.8189
5,5,34.0158
6,6,34.2889
7,7,34.6749
8,8,34.6749
9,9,34.5992


**B. Interpret results**

Run the same parameters in the lossless OPF from problem 1. How do prices and flows change? What is the largest magnitude difference in prices between the solution with losses and the lossless OPF solution?

In [26]:
solution = dcopf_ieee(gens_sens, lines, loads);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e+00, 2e+03]
  Cost   [2e+01, 3e+01]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 3e+02]
Presolving model
46 rows, 47 cols, 140 nonzeros  0s
31 rows, 32 cols, 109 nonzeros  0s
13 rows, 14 cols, 52 nonzeros  0s
Dependent equations search running on 10 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
10 rows, 11 cols, 37 nonzeros  0s
Presolve : Reductions: rows 10(-86); columns 11(-213); elements 37(-167)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.2062012930e-02 Pr: 10(2801.04) 0s
         10     7.0700000000e+03 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model status        : Optimal
Simplex   iterations: 10
Objective value     :  7.

In [27]:
price_diff = solution_lossy.prices.value .- solution.prices.value

14-element Vector{Float64}:
 0.0
 1.0350920960385714
 4.4922994051179685
 4.818925653805774
 4.015780785548817
 4.288894353643887
 4.674927369782399
 4.674927369782399
 4.599175403325763
 4.544032599998282
 4.418692029168582
 4.3134127788679635
 4.332570524201561
 4.482609059183446

In [28]:
flow_diff = solution_lossy.flows.flow .- solution.flows.flow

40-element Vector{Float64}:
  8.836989218932487
  3.0282548638247135
  1.4535375195450797
  1.1623012134132082
  0.8772562269567032
 -0.48424178679367813
 -1.2443513941056068
 -0.04492583294965513
 -0.02577795150711637
  0.070703784456974
  0.04257616332827574
  0.006253302622035051
  0.021874318506469592
  ⋮
  0.02577795150711637
 -0.070703784456974
 -0.04257616332827574
 -0.006253302622035051
 -0.021874318506469592
  0.0
  0.044925832949687106
  0.042576163328321925
  0.028127621128527736
  0.04257616332835035
 -0.006253302622050594
 -0.028127621128509084

Prices increase across all nodes (except node 1, where generator 1 supplies all demand), especially at node 4 (~$5/MWh) due to its large transmission distance from generators, when losses are included. Flows generally increase, because more power is being generated to meet the same load due to transmission losses experienced.

## Question 3 - Security contingencies

Power system operators need to ensure that power is delivered reliably even in the event of unexpected outages (**contingencies**). One common contigency that must be planned for is the loss of a transmission line. The security-constrained OPF (SCOPF) run by operators solves for an optimal dispatch that is simultaneously robust (i.e., feasible) to each of the lines failing individually. This is what is known as **N-1 security**, because we assume that at most one component fails in any given scenario.

In this problem, we will not code a full SCOPF, but rather investigate what happens to the feasibility of our problem when we remove transmission lines.

**A. Setup data**

The following code loads the original dataset (with one row per line) and includes a function `format_lines` that converts this to a format that our solver function can use (duplicating rows for both directions, adding susceptance, etc.).

In [29]:
lines = CSV.read(joinpath(datadir,"Tran14.csv"), DataFrame);
rename!(lines,lowercase.(names(lines)))

function format_lines(lines)
    # create line ids 
    lines.id = 1:nrow(lines);
    # add set of rows for reverse direction with same parameters
    lines2 = copy(lines)
    lines2.f = lines2.fromnode
    lines2.fromnode = lines.tonode
    lines2.tonode = lines2.f
    lines2 = lines2[:,names(lines)]
    append!(lines,lines2)

    # calculate simple susceptance, ignoring resistance as earlier 
    lines.b = 1 ./ lines.reactance
    return(lines)
end

format_lines (generic function with 1 method)

Next:

1. Set the capacity of all lines in the system at 100 MW, except for the line $l_{12}$, which you should set to 200 MW.

2. Create a load dataframe `loads_sens` that increases demands everywhere by 10\%

In [30]:
# Set the capacity of all lines in the system at 100 MW, except for the line12, which you should set to 200 MW.
lines.capacity .= 100
lines[(lines.fromnode .== 1) .& (lines.tonode .== 2), :capacity] .= 200

# Create a load dataframe `loads_sens` that increases demands everywhere by 10%
loads_sens = copy(loads)
loads_sens.demand = loads_sens.demand .* 1.1;


**B. Loop over line contingencies**

Create a dataframe `status` with the `fromnode` and `tonode` columns of `lines`.

Create a [for loop](https://docs.julialang.org/en/v1/manual/control-flow/#man-loops) that iterates over each line in `lines` and:
- sets the reactance to be a very high value, 1e9 (i.e., no power will be transmitted)
- creates a version of the dataframe that our solver function can use via `format_lines`
- runs DCOPF
- stores the solution status in a `opf` column in the corresponding row of the `status` dataframe

Show the `status` results.

In [31]:
# Create a dataframe `status` with the `fromnode` and `tonode` columns of `lines` + empty 'opf' column for storing the solution status
status = DataFrame(
    fromnode = lines.fromnode,
    tonode = lines.tonode,
    opf = Vector{Any}(undef, nrow(lines)),
);

In [32]:
# Create a for loop that iterates over each line in `lines`
for l in 1:nrow(lines)
    lines_copy = copy(lines)
    lines_copy[l,:reactance] = 1e9
    formatted_lines = format_lines(lines_copy)
    try
        solution = dcopf_ieee(gens, formatted_lines, loads_sens)
        status[l,:opf] = solution.status
    catch
        status[l,:opf] = "INFEASIBLE"
    end
end
status

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 96 rows; 224 cols; 204 nonzeros
Coefficient ranges:
  Matrix [1e-07, 2e+03]
  Cost   [2e+01, 2e+01]
  Bound  [0e+00, 0e+00]
  RHS    [4e+00, 3e+02]
Presolving model
46 rows, 47 cols, 140 nonzeros  0s
30 rows, 31 cols, 101 nonzeros  0s
11 rows, 12 cols, 42 nonzeros  0s
Dependent equations search running on 8 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
8 rows, 9 cols, 27 nonzeros  0s
Presolve : Reductions: rows 8(-88); columns 9(-215); elements 27(-177)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -6.3543516829e-05 Pr: 8(424.049) 0s
          9     6.6226433651e+03 0s
Model status        : Infeasible
Simplex   iterations: 9
Objective value     :  6.6224999999e+03
HiGHS run time      :          0.00
Solving LP to try to

Row,fromnode,tonode,opf
,Int64,Int64,Any
1,1,2,INFEASIBLE
2,1,5,OPTIMAL
3,2,3,INFEASIBLE
4,2,4,INFEASIBLE
5,2,5,OPTIMAL
6,3,4,INFEASIBLE
7,4,5,OPTIMAL
8,4,7,OPTIMAL
9,4,9,OPTIMAL


**3. Interpret results**

**Are all of the cases feasible? If not, how many are infeasible?**

No, four out of twenty cases are infeasible.

**Pick two cases where the solution gives a different status. (For our purposes, dual infeasible and primal infeasible are the same.) What is happening here?**

**Given this, do you conclude that the system with the assumed transmission line ratings is secure as-is, or do we need to add more redundancy to the system?**

Let's pick the first two cases. 

1. When l12 fails, the case is infeasible, because it is a critical line connecting generator 1 and 2. Generator 1 cannot supply load beyond node 5 and generator 2 cannot meet system load given its low capacity and line limits.

2. When l15 fails, the case remains feasible, because alternative paths exist connecting node 1 and 5 to reroute supply so that it meets demand given existing line limits.

The system with the assumed transmission line ratings is not N-1 secure as-is. We need to add more redundancy to the system so that when l12, l23, l24, and l34 individually fail, there are alternate routes or sufficient transmission capacity on current routes for supply to meet demand and the system to remain feasible.